# BGE-M3 and Qwen3-Embedding-4B on Kaggle

Run the repository's Stage 08 with a Kaggle GPU. Before starting, attach the **entire `sc-disrupt` project** as a Kaggle Dataset, enable **GPU** and **Internet** in Notebook Settings, and make sure the project dataset contains `data/03_processed/news_full_text/merged_full_text_unique.csv`.

The notebook runs **BAAI/bge-m3** and **Qwen/Qwen3-Embedding-4B** as separate experiments. Within each experiment, article embeddings are computed once and reused to create five-week and three-calendar-month event maps. Each model exports blinded validation pairs for cosine thresholds 0.85, 0.88, 0.90, and 0.92.

In [ ]:
%pip install -q "sentence-transformers>=2.7.0" "transformers>=4.51.0" accelerate pyyaml

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import pandas as pd
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook Settings, then restart.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Locate and copy the attached project

Kaggle input datasets are read-only, so the project is copied to `/kaggle/working/sc-disrupt`. The search fails closed if no project or multiple project copies are found.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/sc_disrupt_work")
STAGE_SCRIPT = Path("scripts/08_filter_irrelevant_news.py")
REQUIRED_INPUT = Path("data/03_processed/news_full_text/merged_full_text_unique.csv")

project_candidates = sorted(
    {path.parent.parent.resolve() for path in INPUT_ROOT.rglob(STAGE_SCRIPT.as_posix())}
)
if len(project_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one attached sc-disrupt project, found {len(project_candidates)}: "
        f"{project_candidates}"
    )
SOURCE_ROOT = project_candidates[0]
if not (SOURCE_ROOT / REQUIRED_INPUT).is_file():
    raise FileNotFoundError(
        f"Required article CSV is missing: {SOURCE_ROOT / REQUIRED_INPUT}"
    )

if not str(WORK_ROOT).startswith("/kaggle/working/"):
    raise RuntimeError(f"Refusing non-writable work path: {WORK_ROOT}")
if WORK_ROOT.is_symlink():
    WORK_ROOT.unlink()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    SOURCE_ROOT,
    WORK_ROOT,
    ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", "*.pyc"),
    dirs_exist_ok=True,
)
print(f"Project source: {SOURCE_ROOT}")
print(f"Writable copy:  {WORK_ROOT}")
print(f"Input size:     {(WORK_ROOT / REQUIRED_INPUT).stat().st_size / 1e6:.1f} MB")

## Run both embedding models

Both models run in FP16. BGE-M3 uses batch size 2, while the 4B Qwen model uses batch size 1 to reduce out-of-memory risk on a 16 GB GPU. Each run writes to a separate directory, so results cannot overwrite one another.

In [ ]:
RUN_ROOT = Path("/kaggle/working/event_embedding_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_RUNS = [
    {"slug": "bge_m3", "model": "BAAI/bge-m3", "batch_size": 2},
    {"slug": "qwen3_embedding_4b", "model": "Qwen/Qwen3-Embedding-4B", "batch_size": 1},
]

for model_run in MODEL_RUNS:
    output_dir = RUN_ROOT / model_run["slug"]
    command = [
        sys.executable,
        str(STAGE_SCRIPT),
        "--input",
        str(WORK_ROOT / REQUIRED_INPUT),
        "--output-dir",
        str(output_dir),
        "--event-model",
        model_run["model"],
        "--event-device",
        "cuda",
        "--event-torch-dtype",
        "float16",
        "--event-batch-size",
        str(model_run["batch_size"]),
    ]
    print(f"\nRunning {model_run['model']}")
    print("Command:", " ".join(command))
    subprocess.run(command, cwd=WORK_ROOT, check=True)
    torch.cuda.empty_cache()

In [ ]:
for model_run in MODEL_RUNS:
    output_root = RUN_ROOT / model_run["slug"]
    event_root = output_root / "event_dedup"
    manifest_path = event_root / "canonical_event_manifest.json"
    event_map_path = event_root / "canonical_event_article_map.csv"
    if not manifest_path.is_file() or not event_map_path.is_file():
        raise RuntimeError(f"Missing event outputs for {model_run['model']}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    event_map = pd.read_csv(event_map_path, nrows=3)
    print(f"\nMODEL: {model_run['model']}")
    print("Provisional threshold:", manifest["selected_threshold_provisional"])
    print("Threshold candidates:", manifest["threshold_candidates_for_human_validation"])
    for profile in ("weekly", "monthly"):
        profile_audit = manifest["profiles"][profile]
        print(
            profile,
            profile_audit["date_window_interpretation"],
            f"events={profile_audit['counts']['events']:,}",
        )
    display(event_map)

## Package results

Download the generated ZIP from the Kaggle file browser. It contains one folder per embedding model. After human validation selects the model and threshold, copy that model's folder contents into `data/03_processed/news_full_text/pre_llm_filter/` in the local project. Do not combine the two event maps.

In [ ]:
ARCHIVE_BASE = Path("/kaggle/working/event_embedding_outputs")
archive_path = Path(
    shutil.make_archive(
        str(ARCHIVE_BASE),
        "zip",
        root_dir=RUN_ROOT,
    )
)
print(f"Download: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / 1e6:.1f} MB")